In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 2.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

import optuna

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD, Adam, RMSprop, Adagrad, Adadelta, Nadam

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
data = pd.read_csv('Salary_Data.csv')

data

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0
...,...,...,...,...,...,...
6699,49.0,Female,PhD,Director of Marketing,20.0,200000.0
6700,32.0,Male,High School,Sales Associate,3.0,50000.0
6701,30.0,Female,Bachelor's Degree,Financial Manager,4.0,55000.0
6702,46.0,Male,Master's Degree,Marketing Manager,14.0,140000.0


In [ ]:
data.describe(include='all')

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
count,6702.000000,6702,6701,6702,6701.000000,6699.000000
unique,NaN,3,7,193,NaN,NaN
top,NaN,Male,Bachelor's Degree,Software Engineer,NaN,NaN
freq,NaN,3674,2267,518,NaN,NaN
mean,33.620859,NaN,NaN,NaN,8.094687,115326.964771
std,7.614633,NaN,NaN,NaN,6.059003,52786.183911
min,21.000000,NaN,NaN,NaN,0.000000,350.000000
25%,28.000000,NaN,NaN,NaN,3.000000,70000.000000
50%,32.000000,NaN,NaN,NaN,7.000000,115000.000000
75%,38.000000,NaN,NaN,NaN,12.000000,160000.000000


In [ ]:
data.isnull().sum()

,0
Age,2
Gender,2
Education Level,3
Job Title,2
Years of Experience,3
Salary,5


In [ ]:
data = data.dropna()

In [ ]:
data['Job_Avg_Exp'] = data.groupby('Job Title')['Years of Experience'].transform('mean')
data['Edu_Avg_Exp'] = data.groupby('Education Level')['Years of Experience'].transform('mean')

/tmp/ipykernel_2576/1964514027.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Job_Avg_Exp'] = data.groupby('Job Title')['Years of Experience'].transform('mean')
/tmp/ipykernel_2576/1964514027.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Edu_Avg_Exp'] = data.groupby('Education Level')['Years of Experience'].transform('mean')


In [ ]:
data = data.drop('Job Title', axis=1)

data.head()

,Age,Gender,Education Level,Years of Experience,Salary,Job_Avg_Exp,Edu_Avg_Exp
0,32.0,Male,Bachelor's,5.0,90000.0,4.449807,5.550926
1,28.0,Female,Master's,3.0,65000.0,4.969697,9.489583
2,45.0,Male,PhD,15.0,150000.0,17.500000,13.920322
3,36.0,Female,Bachelor's,7.0,60000.0,1.428571,5.550926
4,52.0,Male,Master's,20.0,200000.0,20.000000,9.489583


In [ ]:
data['Education Level'].value_counts()

,count
Education Level,
Bachelor's Degree,2265
Master's Degree,1572
PhD,1368
Bachelor's,756
High School,448
Master's,288
phD,1


In [ ]:
data['Education Level'] = np.where(data['Education Level'] == "Bachelor's", "Bachelor's Degree", data['Education Level'])
data['Education Level'] = np.where(data['Education Level'] == "Master's", "Master's Degree", data['Education Level'])
data['Education Level'] = np.where(data['Education Level'] == "phD", "PhD", data['Education Level'])


In [ ]:
data['Education Level'].value_counts()

,count
Education Level,
Bachelor's Degree,3021
Master's Degree,1860
PhD,1369
High School,448


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['Gender'] = le.fit_transform(data['Gender'])
data['Education Level'] = le.fit_transform(data['Education Level'])

In [ ]:
data.head()

,Age,Gender,Education Level,Years of Experience,Salary,Job_Avg_Exp,Edu_Avg_Exp
0,32.0,1,0,5.0,90000.0,4.449807,5.550926
1,28.0,0,2,3.0,65000.0,4.969697,9.489583
2,45.0,1,3,15.0,150000.0,17.500000,13.920322
3,36.0,0,0,7.0,60000.0,1.428571,5.550926
4,52.0,1,2,20.0,200000.0,20.000000,9.489583


In [ ]:
targets = data['Salary']

inputs = data.drop(['Salary'],axis=1)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(inputs)

scaled = scaler.transform(inputs)

inputs_scaled = pd.DataFrame(scaled, columns=inputs.columns)

inputs_scaled

,Age,Gender,Education Level,Years of Experience,Job_Avg_Exp,Edu_Avg_Exp
0,-0.213129,0.892906,-1.011826,-0.510769,-0.811518,-0.695416
1,-0.738393,-1.101320,0.626171,-0.840811,-0.695782,0.381131
2,1.493980,0.892906,1.445169,1.139440,2.093664,1.592177
3,0.312135,-1.101320,-1.011826,-0.180727,-1.484093,-0.695416
4,2.413192,0.892906,0.626171,1.964544,2.650204,0.381131
...,...,...,...,...,...,...
6693,2.019244,-1.101320,1.445169,1.964544,0.884454,1.592177
6694,-0.213129,0.892906,-0.192828,-0.840811,-1.484093,-1.689169
6695,-0.475761,-1.101320,-1.011826,-0.675790,0.578546,-0.743306
6696,1.625296,0.892906,0.626171,0.974419,0.302696,0.431619


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(inputs_scaled, targets, test_size=0.2, random_state=42)

In [ ]:
def create_model(trial):
    model = Sequential()

    model.add(Dense(units=trial.suggest_int('units_layer1', 6, 32), activation='relu'))
    model.add(Dense(units=trial.suggest_int('units_layer2', 6, 32), activation='relu'))
    model.add(Dense(units=1, activation='relu'))

    optimizer_name = trial.suggest_categorical('optimizer', ['adam', 'sgd', 'rmsprop', 'adagrad'])
    learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)

    if optimizer_name == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        optimizer = SGD(learning_rate=learning_rate)
    elif optimizer_name == 'rmsprop':
        optimizer = RMSprop(learning_rate=learning_rate)
    elif optimizer_name == 'adagrad':
        optimizer = Adagrad(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss='mae', metrics=['mae'])

    return model

In [ ]:
from sklearn.metrics import r2_score

def optimal(trial):

    epochs = trial.suggest_int('epochs', 10, 50)
    batch_size = trial.suggest_int('batch_size', 16, 64)

    model = create_model(trial)

    model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)

    y_pred = model.predict(X_test)


    r2 = r2_score(y_test, y_pred)

    return r2

study = optuna.create_study(direction='maximize')
study.optimize(optimal, n_trials=10)

print(f"Ən yaxşı R2 Score: {study.best_trial.value}")
print(f"Ən yaxşı parametrlər: {study.best_trial.params}")

[I 2026-04-18 10:28:58,849] A new study created in memory with name: no-name-23a7613d-61a8-47c2-96c7-5c421c9c41e3
/tmp/ipykernel_2576/2420489113.py:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


[I 2026-04-18 10:29:15,450] Trial 0 finished with value: -4.609472394955122 and parameters: {'epochs': 25, 'batch_size': 18, 'units_layer1': 18, 'units_layer2': 27, 'optimizer': 'rmsprop', 'learning_rate': 4.971962793521218e-05}. Best is trial 0 with value: -4.609472394955122.
/tmp/ipykernel_2576/2420489113.py:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


[I 2026-04-18 10:29:20,647] Trial 1 finished with value: 0.7117198620559075 and parameters: {'epochs': 23, 'batch_size': 62, 'units_layer1': 6, 'units_layer2': 28, 'optimizer': 'sgd', 'learning_rate': 0.001821962202551651}. Best is trial 1 with value: 0.7117198620559075.
/tmp/ipykernel_2576/2420489113.py:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


...

42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


[I 2026-04-18 10:30:35,482] Trial 8 finished with value: -4.611073377771043 and parameters: {'epochs': 25, 'batch_size': 40, 'units_layer1': 18, 'units_layer2': 7, 'optimizer': 'adagrad', 'learning_rate': 0.0029653289526630597}. Best is trial 1 with value: 0.7117198620559075.
/tmp/ipykernel_2576/2420489113.py:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)


42/42 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


[I 2026-04-18 10:30:40,643] Trial 9 finished with value: -4.614220473031328 and parameters: {'epochs': 13, 'batch_size': 30, 'units_layer1': 22, 'units_layer2': 19, 'optimizer': 'adagrad', 'learning_rate': 1.3500218137747507e-05}. Best is trial 1 with value: 0.7117198620559075.


Ən yaxşı R2 Score: 0.7117198620559075
Ən yaxşı parametrlər: {'epochs': 23, 'batch_size': 62, 'units_layer1': 6, 'units_layer2': 28, 'optimizer': 'sgd', 'learning_rate': 0.001821962202551651}


In [ ]:
best_params = study.best_trial.params

best_params

{'epochs': 23,
 'batch_size': 62,
 'units_layer1': 6,
 'units_layer2': 28,
 'optimizer': 'sgd',
 'learning_rate': 0.001821962202551651}

In [ ]:
best_model = Sequential()
best_model.add(Dense(units=best_params['units_layer1'], activation='relu'))
best_model.add(Dense(units=best_params['units_layer2'], activation='relu'))
best_model.add(Dense(1, activation='relu'))

In [ ]:
if best_params['optimizer'] == 'adam':
    best_optimizer = Adam(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'sgd':
    best_optimizer = SGD(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'rmsprop':
    best_optimizer = RMSprop(learning_rate=best_params['learning_rate'])
elif best_params['optimizer'] == 'adagrad':
    best_optimizer = Adagrad(learning_rate=best_params['learning_rate'])


In [ ]:
best_model.compile(optimizer=best_optimizer, loss='mae', metrics=['mae'])

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test):

    model.fit(X_train, y_train, epochs=best_params['epochs'], batch_size=best_params['batch_size'])


    y_train_pred = model.predict(X_train)


    y_test_pred = model.predict(X_test)


    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)



    mae_train = mean_absolute_error(y_train, y_train_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)


    results = pd.DataFrame({
        'Dataset': ['Train', 'Test'],
        'R2 Score (%)': [r2_train * 100, r2_test * 100],
        'Mean Absolute Error (MAE)': [mae_train, mae_test]

    })

    return results

In [ ]:
evaluate(best_model, X_train, y_train, X_test, y_test)

Epoch 1/23
87/87 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 115485.1797 - mae: 115485.1797
Epoch 2/23
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 115484.9922 - mae: 115484.9922
Epoch 3/23
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 115484.7188 - mae: 115484.7188
Epoch 4/23
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 115484.2578 - mae: 115484.2578
Epoch 5/23
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 115483.4375 - mae: 115483.4375
Epoch 6/23
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 115481.8516 - mae: 115481.8516
Epoch 7/23
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 115478.5625 - mae: 115478.5625
Epoch 8/23
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 115469.6406 - mae: 115469.6406
Epoch 9/23
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 115427.8203 - mae: 115427.8203
Epoch 10/23
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 114173.1250 - mae: 114173.1250
Epoch 11/23
87/87 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 41697.7773 - mae: 41697.7773
Epoch 12/23
87/87 ━━━

,Dataset,R2 Score (%),Mean Absolute Error (MAE)
0,Train,72.324515,21013.087600
1,Test,72.506620,21399.947336
